<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week12/day5-6/Copy_of_Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "This is an example of a short sentence using AutoTokenizer"
print(sample_sentence)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

This is an example of a short sentence using AutoTokenizer


In [2]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | this         |  2023
    2 | is           |  2003
    3 | an           |  2019
    4 | example      |  2742
    5 | of           |  1997
    6 | a            |  1037
    7 | short        |  2460
    8 | sentence     |  6251
    9 | using        |  2478
   10 | auto         |  8285
   11 | ##tok        | 18715
   12 | ##eni        | 18595
   13 | ##zer        |  6290
   14 | [SEP]        |   102
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (14, '[SEP]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]'), (21, '[PAD]

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.

[CLS]/[SEP]: [CLS] (index 0) and [SEP] (index 14) are added automatically as special tokens marking the start and end of the sentence; they get their own embeddings and pass through the encoder like normal tokens, with [CLS]'s final hidden state often used as a pooled sentence representation.

Attention mask: Positions with mask value 0 (the [PAD] tokens, indices 15–23) are excluded from self-attention by adding a large negative value to their attention scores before softmax, so real tokens attend only to each other and never to padding.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This is a horrible, bad and negative sentence"
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9998062252998352}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?

Match expectation? Yes — the sentence uses clearly negative words ("horrible", "bad", "negative"), so a NEGATIVE label is expected.

Confidence: The score (~99.98%) is very close to 1, meaning the model is highly confident in its prediction, since the sentiment signal in the text is strong and unambiguous.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        text = text.strip()
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()
        label = self.model.config.id2label[pred_id]
        return {"label": label, "score": probs[pred_id].item()}


In [5]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
#     "TODO: add a clearly positive statement",
            "This movie was amazing and I loved it!",
#     "TODO: add a clearly negative statement"
            "This movie was terrible and I hated it."
]
for text in samples:
    print(text)
    print(analyzer.predict(text))
# ]
# for text in samples:
#     print(text)
#     print(analyzer.predict(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

This movie was amazing and I loved it!
{'label': 'POSITIVE', 'score': 0.9998840093612671}
This movie was terrible and I hated it.
{'label': 'NEGATIVE', 'score': 0.9996681213378906}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        labels = [self.model.config.id2label[p] for p in predictions]

        entities = []
        current_entity = None

        for token, label in zip(tokens, labels):
            if token in self.tokenizer.all_special_tokens:
                continue

            # Merge subword tokens (##) into the previous word
            if token.startswith("##"):
                if current_entity is not None:
                    current_entity["text"] += token[2:]
                continue

            if label.startswith("B-"):
                if current_entity is not None:
                    entities.append(current_entity)
                current_entity = {"text": token, "entity": label[2:]}
            elif label.startswith("I-") and current_entity is not None and label[2:] == current_entity["entity"]:
                current_entity["text"] += " " + token
            else:
                if current_entity is not None:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity is not None:
            entities.append(current_entity)

        return entities


In [8]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
# ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
# ner.recognize(sample_text)
ner = BERTNamedEntityRecognizer()
sample_text = "Elon Musk founded SpaceX and later moved to Austin, Texas."
ner.recognize(sample_text)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Elon Musk', 'entity': 'PER'},
 {'text': 'SpaceX', 'entity': 'ORG'},
 {'text': 'Austin', 'entity': 'LOC'},
 {'text': 'Texas', 'entity': 'LOC'}]

**Exercise 5**

| Category | BERT | GPT |
|---|---|---|
| Architecture | Encoder-only, bidirectional self-attention | Decoder-only, causal (left-to-right) self-attention |
| Primary purpose | Understand/represent text (context encoding) | Generate text (next-token prediction) |
| Typical use cases | Classification, NER, embeddings, Q&A extraction | Text generation, chat, summarization, completion |
| Strengths | Deep bidirectional context, strong for understanding tasks | Fluent generation, few-shot flexibility |
| Weaknesses | Can't generate text natively | No bidirectional context, can hallucinate |

**Exercise 6**

- **Encoding queries/documents:** BERT converts the query and each candidate document/passage into a fixed-size dense vector (often the [CLS] embedding or a pooled average), capturing their semantic meaning rather than just keywords.

- **Storing/searching embeddings:** These vectors are stored in a vector database (e.g., FAISS, Pinecone, Weaviate), which indexes them for fast similarity search — at query time, the query embedding is compared against stored document embeddings using cosine similarity or dot product to find the closest matches.

- **Passing to a generative model:** The top-k retrieved passages are inserted into the prompt as context alongside the original query, and a generative model like GPT then produces an answer grounded in that retrieved text.

- **Example application:** A customer support chatbot that retrieves relevant help-center articles (via BERT embeddings) and uses GPT to generate a natural-language answer citing those articles, keeping responses accurate and up to date without retraining the model.